In [1]:
import json
import pandas as pd
import os
import requests
import numpy as np
import datetime

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas as gpd

In [4]:
api_key = "J6SHf5WXbL1NeF0EznTbatA9wknGGfzvp0Xeuk7U"

In [5]:
def get_api_url(weather_year, offset=0):
    return f"https://api.eia.gov/v2/electricity/rto/region-data/data/?frequency=hourly&data[0]=value&start={weather_year}-01-01T00&end={weather_year + 1}-01-01T00&sort[0][column]=period&sort[0][direction]=desc&offset={offset}&length=5000&api_key={api_key}&facets[type][]=D"

hourly_demand_list = []

for weather_year in range(2019, 2025):
    print(f"Getting regional hourly demand for {weather_year}...")

    dfs = []
    num_responses = int(requests.get(get_api_url(weather_year)).json()['response']['total'])
    len_df = 0
    i = 0
    while len_df < num_responses:
        for _ in range(5):
            try:
                response = requests.get(get_api_url(weather_year, i*5000))
                response.raise_for_status()
                break
            except:
                print(f"Got status code {response.status_code}. Retrying API call.")
        
        df = pd.DataFrame.from_records(response.json()['response']['data'])
    
        i += 1
        len_df += len(df)
        dfs.append(df)
    
    hourly_demand = (
        pd.concat(dfs)
        .drop_duplicates()
        .sort_values("period")
        .reset_index(drop=True)
    )
    hourly_demand['timestamp'] = pd.to_datetime(hourly_demand['period'])
    hourly_demand_list.append(hourly_demand)

hourly_regional_demand = pd.concat(hourly_demand_list, ignore_index=True)

Getting regional hourly demand for 2019...
Getting regional hourly demand for 2020...
Getting regional hourly demand for 2021...
Getting regional hourly demand for 2022...
Getting regional hourly demand for 2023...
Getting regional hourly demand for 2024...


In [6]:
def get_api_url(weather_year, offset=0):
        return f"https://api.eia.gov/v2/electricity/rto/region-sub-ba-data/data/?frequency=hourly&data[0]=value&start={weather_year}-01-01T00&end={weather_year+1}-01-01T00&sort[0][column]=period&sort[0][direction]=desc&offset={offset}&length=5000&api_key={api_key}"

hourly_demand_list = []

for weather_year in range(2019, 2025):
    print(f"Getting subregional hourly demand for {weather_year}...")

    dfs = []
    num_responses = int(requests.get(get_api_url(weather_year)).json()['response']['total'])
    len_df = 0
    i = 0
    while len_df < num_responses:
        for _ in range(5):
            try:
                response = requests.get(get_api_url(weather_year, i*5000))
                response.raise_for_status()
                break
            except:
                print(f"Got status code {response.status_code}. Retrying API call.")
        
        df = pd.DataFrame.from_records(response.json()['response']['data'])
    
        i += 1
        len_df += len(df)
        dfs.append(df)
    
    hourly_demand = (
        pd.concat(dfs)
        .drop_duplicates()
        .sort_values("period")
        .reset_index(drop=True)
    )
    hourly_demand['timestamp'] = pd.to_datetime(hourly_demand['period'])
    hourly_demand_list.append(hourly_demand)

hourly_subregional_demand = pd.concat(hourly_demand_list, ignore_index=True)

Getting subregional hourly demand for 2019...
Getting subregional hourly demand for 2020...
Getting subregional hourly demand for 2021...
Getting subregional hourly demand for 2022...
Getting subregional hourly demand for 2023...
Got status code 502. Retrying API call.
Getting subregional hourly demand for 2024...


In [7]:
def get_api_url(weather_year, offset=0):
    return f"https://api.eia.gov/v2/electricity/rto/region-data/data/?frequency=hourly&data[0]=value&start={weather_year}-01-01T00&end={weather_year+1}-01-01T00&sort[0][column]=period&sort[0][direction]=desc&offset={offset}&length=5000&api_key={api_key}&facets[type][]=DF"

hourly_forecast_list = []

for weather_year in range(2019, 2025):
    print(f"Getting regional hourly forecasts for {weather_year}...")

    dfs = []
    num_responses = int(requests.get(get_api_url(weather_year)).json()['response']['total'])
    len_df = 0
    i = 0
    while len_df < num_responses:
        for _ in range(5):
            try:
                response = requests.get(get_api_url(weather_year, i*5000))
                response.raise_for_status()
                break
            except:
                print(f"Got status code {response.status_code}. Retrying API call.")
        
        df = pd.DataFrame.from_records(response.json()['response']['data'])
    
        i += 1
        len_df += len(df)
        dfs.append(df)
    
    hourly_forecast = (
        pd.concat(dfs)
        .drop_duplicates()
        .sort_values("period")
        .reset_index(drop=True)
    )
    hourly_forecast['timestamp'] = pd.to_datetime(hourly_forecast['period'])
    hourly_forecast_list.append(hourly_forecast)

hourly_regional_forecast = pd.concat(hourly_forecast_list, ignore_index=True)

Getting regional hourly forecasts for 2019...
Getting regional hourly forecasts for 2020...
Getting regional hourly forecasts for 2021...
Got status code 502. Retrying API call.
Getting regional hourly forecasts for 2022...
Getting regional hourly forecasts for 2023...
Getting regional hourly forecasts for 2024...


In [9]:
f = open('../data/eia_load_profiles/raw/EBA-pre2019.txt', 'r')
lines = [json.loads(x) for x in f.readlines()]
demand_lines = [line for line in lines if line['series_id'].endswith('.D.H')]
forecast_lines = [line for line in lines if line['series_id'].endswith('.DF.H')]

regional_demand_df_list = []
subregional_demand_df_list = []
for line in demand_lines:
    df = pd.DataFrame(line['data'], columns=['timestamp', 'value'])
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = (
        df.loc[df.timestamp.dt.year >= 2016]
        .sort_values('timestamp')
    )
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

    respondent_info = line['series_id'].split('EBA.')[1].split('-')
    ba = respondent_info[0]
    subba = respondent_info[1].split('.')[0]

    if subba == 'ALL':
        df['respondent'] = ba
        regional_demand_df_list.append(df)
    else:
        df['subba'] = subba
        df['parent'] = ba
        subregional_demand_df_list.append(df)

regional_forecast_df_list = []
for line in forecast_lines:
    df = pd.DataFrame(line['data'], columns=['timestamp', 'value'])
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = (
        df.loc[df.timestamp.dt.year >= 2016]
        .sort_values('timestamp')
    )
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df['respondent'] = line['series_id'].split('EBA.')[1].split('-')[0]
    regional_forecast_df_list.append(df)

In [19]:
df_ba = pd.concat(regional_demand_df_list, ignore_index=True)
df_subba = pd.concat(subregional_demand_df_list, ignore_index=True)
df_ba_forecast = pd.concat(regional_forecast_df_list, ignore_index=True)

In [25]:
eia_930_ref_bas = pd.read_excel('data/EIA930_Reference_Tables.xlsx', sheet_name='BAs')
ba_codes = eia_930_ref_bas['BA Code'].tolist()

df_ba = (
    pd.concat([df_ba, hourly_regional_demand], ignore_index=True)
    [['timestamp', 'respondent', 'value']]
)
df_ba = df_ba.loc[df_ba.respondent.isin(ba_codes)]

df_subba = (
    pd.concat([df_subba, hourly_subregional_demand], ignore_index=True)
    [['timestamp', 'subba', 'value']]
)

df_ba_forecast = (
    pd.concat([df_ba_forecast, hourly_regional_forecast], ignore_index=True)
    [['timestamp', 'respondent', 'value']]
)
df_ba_forecast = df_ba_forecast.loc[df_ba_forecast.respondent.isin(ba_codes)]

In [34]:
for year in df_ba['timestamp'].dt.year.unique():
    _df_ba = df_ba.loc[df_ba.timestamp.dt.year == year].sort_values('timestamp')
    _df_ba.to_csv(
        f"../data/eia_load_profiles/{year}_hourly_demand_by_rto.csv",
        index=False
    )

In [35]:
for year in df_subba['timestamp'].dt.year.unique():
    _df_subba = df_subba.loc[df_subba.timestamp.dt.year == year].sort_values('timestamp')
    _df_subba.to_csv(
        f"../data/eia_load_profiles/{year}_hourly_demand_by_subregion.csv",
        index=False
    )

In [36]:
for year in df_ba_forecast['timestamp'].dt.year.unique():
    _df_ba_forecast = df_ba_forecast.loc[df_ba_forecast.timestamp.dt.year == year].sort_values('timestamp')
    _df_ba_forecast.to_csv(
        f"../data/eia_load_profiles/{year}_hourly_forecast_by_rto.csv",
        index=False
    )